In [1]:
import numpy as np
from scipy import stats
import pandas as pd
import yfinance
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
data = pd.read_csv('/Users/davidmikota/Downloads/SPX_filtered(in).csv')
data.head()

/var/folders/mn/hslfkv6s35g3fwkv7ydgx6fw0000gn/T/ipykernel_11322/1239139428.py:1: DtypeWarning: Columns (40) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/Users/davidmikota/Downloads/SPX_filtered(in).csv')


,Unnamed: 0,Date,SecurityID,Strike,Expiration,CallPut,BestBid,BestOffer,LastTradeDate,Volume,...,DateIndex,Implied_Price,BizDays,VolPred,Log return prices,meu hat,Historic Volatility,d_plus,d_minus,St = 613.4
0,1,1/22/1996,108105,590,2/16/1996,C,24.375,24.750,1/22/1996,860,...,1,611.73251,19,9.939235,0.023818,0.006424,0.195242,8.682168,7.831126,st+1= 647.98
1,2,1/22/1996,108105,595,2/16/1996,C,19.750,20.250,1/22/1996,32,...,1,611.73251,19,9.939235,0.023818,0.009533,0.195242,126.343903,125.492861,NaN
2,3,1/22/1996,108105,600,2/16/1996,C,15.375,16.125,1/22/1996,816,...,1,611.73251,19,9.939235,0.023818,NaN,0.195242,126.334070,125.483028,=NORM.S.DIST(AM2)
3,4,1/22/1996,108105,605,2/16/1996,C,11.750,12.500,1/22/1996,24,...,1,611.73251,19,9.939235,0.023818,0.011615,0.195242,126.324318,125.473277,NaN
4,5,1/22/1996,108105,610,2/16/1996,C,8.750,9.250,1/22/1996,2440,...,1,611.73251,19,9.939235,0.023818,NaN,0.195242,126.314647,125.463605,NaN


From the Oxford Man we have the realised variance which is defined as the sum of the squared returns $$ rv5(t) = \sum_i^n(r_{t,i})^2$$. The realised variance has been calculated from intradaily 5 minute intervals. Where $t$ is the specific day and the log returns are $$r_{t,i} = log(P_{t,i+5}/P_{t,i})$$ 

Daily Volatility is defined as $$ \sigma_{daily} = \sqrt{rv5}$$ While yearly historic volatility is 
$$ \sigma_{annual} = \sigma_{daily} \sqrt{252} $$

In [284]:
### First renaiming the date column then changing it to a date time then calculating the yearly historic volatility in a pd.dataframe


realised_variance = pd.read_csv('/Users/davidmikota/Downloads/historical_vol.csv')# start:2000-01-03, end: 2020-03-31

realised_variance = realised_variance.rename(columns = {"Unnamed: 0" : "Date"})

realised_variance['Date'] = pd.to_datetime(realised_variance['Date'])

vol_hist = pd.DataFrame({
    'Date': realised_variance['Date'] ,
'Historical Volatility': np.sqrt(realised_variance['rv5'] * 252) })



### The data starts in 1996 however the historical volatility data starts in 2000. As as result we will need to
### calculate the historical volatility from the standard deviation of the log return and standardise it.

In [7]:
ticker_data = yfinance.download(tickers = '^SPX', start = '1996-01-22', end= '2000-01-02')
ticker_data

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^SPX,^SPX,^SPX,^SPX,^SPX
Date,,,,,
1996-01-22,613.400024,613.450012,610.950012,611.830017,398040000
1996-01-23,612.789978,613.400024,610.650024,613.400024,416910000
1996-01-24,619.960022,619.960022,612.789978,612.789978,476380000
1996-01-25,617.030029,620.150024,616.619995,619.960022,453270000
1996-01-26,621.619995,621.700012,615.260010,617.030029,385700000
...,...,...,...,...,...
1999-12-27,1457.099976,1463.189941,1450.829956,1458.339966,722600000
1999-12-28,1457.660034,1462.680054,1452.780029,1457.089966,655400000


Historic volatility can also be calculated as follows

$$ \sigma_{historic annualised} = \sqrt{252}*\sigma_{daily_log_returns} $$

In [288]:
log_returns = np.log(ticker_data['Close']/ticker_data['Open']) # The open of today is the same as the close of yesterday
daily_vol = np.std(log_returns, axis = 0)

calculated_historic_vol = daily_vol.values * np.sqrt(252) * np.ones(997)
calculated_historic_vol # 0.173937 annualised historic vol



calculated_historic_vol = pd.DataFrame(data = 
                                      {'Date': ticker_data.index,
                                      'Historical Volatility': calculated_historic_vol})



In [290]:
calculated_historic_vol

,Date,Historical Volatility
0,1996-01-22,0.173937
1,1996-01-23,0.173937
2,1996-01-24,0.173937
3,1996-01-25,0.173937
4,1996-01-26,0.173937
...,...,...
992,1999-12-27,0.173937
993,1999-12-28,0.173937
994,1999-12-29,0.173937
995,1999-12-30,0.173937


## Merging both historical volatilities so that it covers the whole time interval in which we have options data

In [12]:
vol_hist

,Date,Historical Volatility
0,2000-01-03,0.188376
1,2000-01-04,0.237657
2,2000-01-05,0.281423
3,2000-01-06,0.181560
4,2000-01-07,0.153599
...,...,...
5074,2020-03-25,0.577690
5075,2020-03-26,0.524757
5076,2020-03-27,0.378964
5077,2020-03-30,0.295328


In [292]:
combined_hist_vol = pd.merge(calculated_historic_vol, vol_hist, how = 'outer')
combined_hist_vol.head()

,Date,Historical Volatility
0,1996-01-22,0.173937
1,1996-01-23,0.173937
2,1996-01-24,0.173937
3,1996-01-25,0.173937
4,1996-01-26,0.173937


 ### For the implied volatility of the SP500 we will use CBOE Volatility Index which is the index ^VIX, 

In [15]:
implied_vol = yfinance.download(tickers = '^VIX', start = '1996-01-22', end = '2020-03-31')
implied_vol = implied_vol['Close']
implied_vol

[*********************100%***********************]  1 of 1 completed


Ticker,^VIX
Date,
1996-01-22,13.340000
1996-01-23,13.560000
1996-01-24,12.540000
1996-01-25,12.940000
1996-01-26,12.000000
...,...
2020-03-24,61.669998
2020-03-25,63.950001
2020-03-26,61.000000


In [382]:


closes_spx = yfinance.download(tickers = '^SPX', start = '1996-01-22', end = '2020-03-21')

closes_spx = pd.DataFrame({'Date': pd.to_datetime(closes_spx.index),
                                   'SPX Closes': closes_spx['Close'].values.squeeze()})


[*********************100%***********************]  1 of 1 completed


In [384]:
closes_spx

,Date,SPX Closes
0,1996-01-22,613.400024
1,1996-01-23,612.789978
2,1996-01-24,619.960022
3,1996-01-25,617.030029
4,1996-01-26,621.619995
...,...,...
6078,2020-03-16,2386.129883
6079,2020-03-17,2529.189941
6080,2020-03-18,2398.100098
6081,2020-03-19,2409.389893


The data from yahoo finance for which we need because of the VIX(implied volatility) has a different amount of observations during the same time interval as the data set rv5 originally from r. Upon closer inspection of the missing data values we see that the days appear to be random. As a result, for simplicity I will remove the days which coincide with missing values from the dataframe.

In [129]:
len(realised_future_vol) #6083 # start 96-01-22 # ends 2020-03-31
len(implied_vol) # 6089
# len(combined_hist_vol) #6076  # start 96-01-22 # ends 2020-03-31
# len(implied_vol.index) # 6089 # start 96-01-22 # ends 2020-03-31

6089

In [123]:
# check which dates are missing between the two

hist_dates = set(combined_hist_vol['Date'])
impl_dates = set(implied_vol.index)

# dates in implied_vol but not in combined_hist_vol
missing_in_hist = sorted(impl_dates - hist_dates)

# dates in combined_hist_vol but not in implied_vol
missing_in_impl = sorted(hist_dates - impl_dates)

print("Missing in hist:")
print(missing_in_hist)

print("\nMissing in implied:")
print(missing_in_impl)

Missing in hist:
[Timestamp('2000-03-17 00:00:00'), Timestamp('2001-03-08 00:00:00'), Timestamp('2001-03-21 00:00:00'), Timestamp('2001-10-08 00:00:00'), Timestamp('2002-10-31 00:00:00'), Timestamp('2003-01-17 00:00:00'), Timestamp('2003-01-21 00:00:00'), Timestamp('2004-01-12 00:00:00'), Timestamp('2004-01-13 00:00:00'), Timestamp('2004-10-12 00:00:00'), Timestamp('2018-07-23 00:00:00'), Timestamp('2019-06-24 00:00:00'), Timestamp('2019-11-18 00:00:00'), Timestamp('2019-12-16 00:00:00')]

Missing in implied:
[Timestamp('2020-03-31 00:00:00')]


In [386]:
# check which dates are missing between the two

reali_dates = set(closes_spx['Date'])
impl_dates = set(implied_vol.index)

# dates in implied_vol but not in combined_hist_vol
missing_in_reali= sorted(impl_dates - reali_dates)

# dates in combined_hist_vol but not in implied_vol
missing_in_impl = sorted(reali_dates - impl_dates)

print("Missing in reali:")
print(missing_in_reali)

print("\nMissing in implied:")
print(missing_in_impl)

Missing in reali:
[Timestamp('2020-03-23 00:00:00'), Timestamp('2020-03-24 00:00:00'), Timestamp('2020-03-25 00:00:00'), Timestamp('2020-03-26 00:00:00'), Timestamp('2020-03-27 00:00:00'), Timestamp('2020-03-30 00:00:00')]

Missing in implied:
[]


I looked up online that these dates have closing values for the SPX so I just put them in manually. As a result there is no longer any dates in which we have missing values between the SPX and VIX indexes.

In [396]:
#realised_future_vol.loc[realised_future_vol['Date'] == '2020-03-20 00:00:00' ,'Realised Volatility']

new_dates = pd.DataFrame({ 'Date': pd.to_datetime([
        '2020-03-23 00:00:00',
        '2020-03-24 00:00:00',
        '2020-03-25 00:00:00',
        '2020-03-26 00:00:00',
        '2020-03-27 00:00:00',
        '2020-03-30 00:00:00'
    ]),
        'SPX Closes' : [
        2237.40,
        2447.33,
        2475.56,
        2630.07,
        2541.47,
        2626.65
    ]}
    )

closes_spx = pd.concat([closes_spx, new_dates], axis = 0, ignore_index = False)
closes_spx = closes_spx.sort_values('Date', ascending = True)

closes_spx
# 


,Date,SPX Closes
0,1996-01-22,613.400024
1,1996-01-23,612.789978
2,1996-01-24,619.960022
3,1996-01-25,617.030029
4,1996-01-26,621.619995
...,...,...
4,2020-03-27,2541.470000
4,2020-03-27,2541.470000
5,2020-03-30,2626.650000
5,2020-03-30,2626.650000


In [394]:
filtered_implied_vol = implied_vol[ ~implied_vol.index.isin(missing_in_hist)] # checks if the current implied vol index is in missing hist
# tilda flips the False to True

#Filtered implied vol is 6075 x 1 

filtered_combined_hist_vol = combined_hist_vol[ ~combined_hist_vol.Date.isin(missing_in_impl)] # 6075 x 2 
filtered_realised_future_vol = closes_spx[ ~closes_spx.Date.isin(missing_in_hist)]# 6075 x 2  


Now calculate realised future volatility, its realised because its the actual volatility of the SP500. 

$$ 

In [ ]:
log_returns_spx = np.log(closes_spx['SPX Closes']/ closes_spx['SPX Closes'].diff(1) )

var_returnsSpx = []

for i in range(0,N)

In python in order to use least squares we need to create a common dataframe.



In [342]:
vol_df = pd.DataFrame(
    {'Date' : filtered_implied_vol.index, # length 6075
    'Implied Volatility' : filtered_implied_vol.values.squeeze(), # len 6075
    'Historic Volatility' : filtered_combined_hist_vol['Historical Volatility'].iloc[:-1].values,
    'Realised Future Volatility' : filtered_realised_future_vol['Realised Volatility'].values})

vol_df.set_index('Date')
vol_df['Implied Volatility'] = vol_df['Implied Volatility']/100


In [376]:
np.min(vol_df['Realised Future Volatility'])

612.7899780273438

Using statsmodel for linear regression.



In [338]:
mod = smf.ols(formula = 'np.log(Q("Realised Future Volatility")) ~ np.log(Q("Historic Volatility")) + np.log(Q("Implied Volatility"))', data = vol_df)
res = mod.fit()

sigma_hat = np.exp(res.predict())
sigma_tilde = sigma_hat * np.exp((1-res.rsquared)*np.var(np.log(sigma_hat)))
sigma_hat


array([1291.87082618, 1289.68663389, 1300.16230212, ...,  796.26170172,
        870.25941393,  950.37301864])

In [299]:
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                       OLS Regression Results                                      
===================================================================================================
Dep. Variable:     np.log(Q("Realised Future Volatility"))   R-squared:                       0.271
Model:                                                 OLS   Adj. R-squared:                  0.271
Method:                                      Least Squares   F-statistic:                     1131.
Date:                                     Wed, 27 May 2026   Prob (F-statistic):               0.00
Time:                                             12:50:46   Log-Likelihood:                -1697.5
No. Observations:                                     6075   AIC:                             3401.
Df Residuals:                                         6072   BIC:                             3421.
Df Model:                                                2                                         
Covariance Type:                                 nonrobust                                         
====================================================================================================
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            6.4380      0.020    328.854      0.000       6.400       6.476
np.log(Q("Historic Volatility"))    -0.2958      0.013    -22.121      0.000      -0.322      -0.270
np.log(Q("Implied Volatility"))     -0.1034      0.021     -5.040      0.000      -0.144      -0.063
==============================================================================
Omnibus:                      318.768   Durbin-Watson:                   0.084
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              386.145
Skew:                           0.546   Prob(JB):                     1.41e-84
Kurtosis:                       3.576   Cond. No.                         18.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [269]:

np.exp(-0.2797) #0.756
np.exp(-0.1304) #0.8778

0.8777442629864621

In [275]:
len(sigma_tilde)

6075

In [277]:
6075/252

24.107142857142858

$$R_i^{year} = S_i/S_{i-{21}} -1 $$. There are 12 $R_i^{year}$ per year, $$\tilde{R_i} = \frac{R_i - \bar{\mu}\Delta_i}{\bar{\sigma}\sqrt{\Delta_i}}$$

$$\hat{R_i} = \tilde{R_i}\tilde{\sigma}_{[t,T]}\sqrt{\Delta}+ \bar{\mu}\Delta$$

$\tilde{R_i}$ is the annualised scaled historic returns of the month $i$. As a result there are 12 of $R_i$ per year since 12 months per year. While there are 6075 $\tilde{sigma_j}$ which represent the forecasted annualised volatility of given day j. To make the $tilde{sigma_j}$ on the same scale that is on a monthly basis I will calculate the total forecasted annualised variance of a given month $i$ and will square root it so that it becomes the forecasted annualised volatility for month i.

$$ \tilde{\sigma_i} =  \sqrt{\sum_{j=1}^{21}  \tilde{\sigma^2_j}} $$

$$ \tilde{\sigma^2_j} = (\tilde{\sigma})^2 $$.

Where $i$ represents a given month and $j$ represents a given day


In [330]:
daily_annualised_forecasted_var = sigma_tilde**2
daily_annualised_forecasted_var = pd.DataFrame({'Date':filtered_implied_vol.index
                                                ,'daily_annualised_forecasted_var': daily_annualised_forecasted_var  })
sigma_tilde
#monthly_annualised_forecasted_vol = sum

array([3475.47948211, 3469.60341825, 3497.78577924, ..., 2142.15783082,
       2341.23155034, 2556.75866326])

In [26]:
Spx = yfinance.download(tickers = '^SPX', start = '1996-01-22',end = '2020-03-31' )
Spx['Close']
rs = [] # returns - 1

for i in range(21,len(Spx), 21):
    try:
        rs.append((Spx.iloc[i, 0] / Spx.iloc[i-21, 0]) -1)
    except: 
        break



mu_bar = np.mean(rs)
sigma_bar = np.std(rs)
delta = 21/252

r_tilde = (rs - mu_bar*delta )/(sigma_bar*np.sqrt(delta))

r_hat = r_tilde* sigma_tilde *np.sqrt(delta) + mu_bar*delta

[*********************100%***********************]  1 of 1 completed


ValueError: operands could not be broadcast together with shapes (289,) (6055,) 

In [ ]:
#len(r_tilde) #289
#len(sigma_tilde) # 6055

### We can now calculate the log realised volatility using regression. Specifically the equation is

$$ log(\sigma_{t,T}) = \beta_{imp}*log(\sigma^{imp}_t)+ \beta_{hist}*log(\sigma^{hist}_t) + \epsilon_{t,T} $$

In [ ]:
dat = sm.datasets.get_rdataset('iris').data
dat.head()
results = smf.ols('np.log(Q("Sepal.Length")) ~ Q("Sepal.Width")', data = dat).fit()
print(results.summary())


results = smf.ols('np.log(realised_vol)~ np.log(vol_hist) + np.log(implied_vol_2nd_date_range)').fit()




In [ ]:
data = pd.read_csv('/Users/davidmikota/Downloads/SPX_filtered(in).csv')
data.tail()

In [ ]:
data.tail()

In [ ]:
hist_vol_data = pd.read_csv('/Users/davidmikota/Downloads/historical_vol.csv')

In [ ]:
call = data[data['CallPut'] == 'C']
put = data[data['CallPut'] == 'P']

In [ ]:
call_r = call['Rate']/100
put_r = put['Rate']/100

log_return_call = np.log(call['PriceExp']/ call['PriceDate']) # note that this is actually ln
log_return_put  = np.log(put['PriceExp']/put['PriceDate'])

In [ ]:
mean_rt_call = log_return_call.mean()
mean_rt_put = log_return_put.mean()

std_rt_call = log_return_call.std()
std_rt_put = log_return_put.std()




In [ ]:
# Historical volatility

hist_vol_call = std_rt_call* np.sqrt(252)
hist_vol_put = std_rt_put* np.sqrt(252)


In [ ]:
hist_vol_call

In [ ]:


def BS_call(S,E,T,r,sigma):
    # S: current stock price
    # E: strike price
    # T: time to expiry in years (T - t)
    # r: risk-free interest rate (annual)
    # sigma: volatility (annualized)
    d1 = (np.log(S / E) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    

    return(S * stats.norm.cdf(d1) - E*np.exp(-r*T) * stats.norm.cdf(d2))

def delta(S,E,T,r,sigma):
    d1 = (np.log(S / E) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return stats.norm.cdf(d1) 


In [ ]:
## Call

Strikes_call =call['Strike']
Underlying_prices_call =  call['PriceDate']
sigma_call = hist_vol_call
T_call = call['BizDays']/252

## Put

Strikes_put = put['Strike']
Underlying_prices_put =  put['PriceDate']
sigma_put = hist_vol_put
T_put= put['BizDays']/252




In [ ]:
call[['Strike', 'PriceDate', 'PriceExp', 'BizDays', 'Rate']].head()

In [ ]:
BS_call(Strikes_call,Underlying_prices_call, T_call, call_r, sigma_call)

future realised returns for day 21 to 42 is $x_21 = \sum_{21}^{42} \log(s_{i+1}/s_{i})/12$

vol of future realised returns std(x_21)= $tilde(sigma_{21})$

2) $VIX_21$ implied volatility for days 21 to 42
3) historic_21 is historic vol of 0 to 21 days

$log(tilde(sigma_{21})) = \beta0*log(VIX_21) + \beta1* log(Historic_21)$

In [5]:
x= 5